In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
pd.set_option('display.max_rows', None)


import os
from utils import loadexp, load_analysis
from activations import LeakyReLU
import metrics
from modules import AE, SAE, SBAE, SOAE
from blocks import Normalizer
from NestedPOD import NestedPOD

### Paths

In [ ]:
datapath = os.path.join('..', 'data')
resultspath = os.path.join('..', 'results')

### Utilities

In [ ]:
# Function to get data
def get_exp_rod(data_id):
    meshpath = os.path.join(datapath, 'rod_mesh.xml')
    trainval_datapath = os.path.join(
        datapath, 
        'rod_snapshots_sample_size_robustness_trainval_id' \
            + str(data_id) + '.npz'
    )
    test_datapath = os.path.join(
        datapath, 
        'rod_snapshots_sample_size_robustness_test.npz'
    )
    hidden_dims = (15, 7, 5)
    exp_rod = {
        'meshpath' : meshpath, 
        'trainval_datapath' : trainval_datapath, 
        'test_datapath' : test_datapath, 
        'hidden_dims' : hidden_dims
    }
    return exp_rod

# Architecture setup
def loadexpwrapper(meshpath, datapath, hidden_dims, split):
    data_split, mesh = loadexp(meshpath, datapath, split)
    data_split_no_none = [elem for elem in data_split if elem is not None]
    n0 = data_split_no_none[0].shape[1]
    red_dims = (n0, ) + hidden_dims
    return data_split, mesh, red_dims


### Raw table visualizations

In [ ]:
# Load and display all table
def get_test_error_raw_analysis():
    test_analysis_raw = load_analysis(
        os.path.join(resultspath, 'analyses/sample_size_robustness_analysis.obj')
    )
    test_analysis_raw = pd.DataFrame(test_analysis_raw)
    test_analysis_raw['training_efficiency'] = test_analysis_raw['elapsed_time'] / test_analysis_raw['epochs']
    return test_analysis_raw

### Extract relevant columns for enhanced visualization

In [ ]:
def extract_info_test_error_analysis(
    raw_test_error_analysis : dict
):
    
    # Extract unique values
    ns_trainval_unique = np.unique(raw_test_error_analysis['ns_trainval'])
    initialization_unique = np.unique(raw_test_error_analysis['initialization'])
    model_name_unique = np.unique(raw_test_error_analysis['model_name'])

    # Initialize dictionary to store raw_test_error_analysis results
    test_error_analysis = dict()
    for model_name in model_name_unique:
        test_error_analysis[model_name] = dict()
        for initialization in initialization_unique:
            test_error_analysis[model_name][initialization] = dict()
            test_error_analysis[model_name][initialization]['ns_trainval'] = list()
            test_error_analysis[model_name][initialization]['MSE_med'] = list()
            test_error_analysis[model_name][initialization]['MSE_min'] = list()
            test_error_analysis[model_name][initialization]['MSE_max'] = list()

    # Fill dictionary and print information 
    for model_name in model_name_unique:
        cond_model_name = (raw_test_error_analysis['model_name'] == model_name)
        for initialization in initialization_unique:
            cond_initialization = (raw_test_error_analysis['initialization'] == initialization)
            for ns_trainval in ns_trainval_unique:
                cond_ns_trainval = (raw_test_error_analysis['ns_trainval'] == ns_trainval)
                curr_mse_all_dataids = raw_test_error_analysis['mse'][cond_ns_trainval * cond_initialization * cond_model_name]
                test_error_analysis[model_name][initialization]['ns_trainval'].append(
                    ns_trainval
                )
                test_error_analysis[model_name][initialization]['MSE_med'].append(
                    np.median(curr_mse_all_dataids)
                )
                test_error_analysis[model_name][initialization]['MSE_min'].append(
                    np.min(curr_mse_all_dataids)
                )
                test_error_analysis[model_name][initialization]['MSE_max'].append(
                    np.max(curr_mse_all_dataids)
                )
                print(
                    (
                        model_name + '\t' + \
                        str(ns_trainval) + '\t' + \
                        str(initialization) + '\t' + \
                        '%1.3e' % np.mean(curr_mse_all_dataids) + '\t' + \
                        '%1.3e' % np.std(curr_mse_all_dataids)
                    ).expandtabs(15)
                )

    return test_error_analysis

### Computation of POD error

In [ ]:
# Function to compute POD error with
def get_pod_error(utrain, utest, N):
    bias = utrain.mean(axis = 0)
    V = torch.linalg.svd((utrain - bias).T)[0]
    VN = V[:,:N]
    uproj = (utest - bias) @ VN @ VN.T + bias
    err_pod = metrics.mse(utrue = utest, upred = uproj).item()

    return err_pod

# Computation of POD error
def get_pod_error_analysis(
    raw_test_error_analysis : dict
):

    # Get values
    ns_train_unique = np.unique(raw_test_error_analysis['ns_train'])
    ns_val_unique = np.unique(raw_test_error_analysis['ns_val'])

    # Initialize dict
    pod_error_analysis = dict()
    pod_error_analysis['test'] = np.zeros((5,4))
    pod_error_analysis['train'] = np.zeros((5,4))
    pod_error_analysis['ns_trainval'] = np.zeros((5,4))

    for data_id in range(1,6):
        for split_idx, (train_split, val_split) in enumerate(
            zip(ns_train_unique, ns_val_unique)
        ):
            trainval_split = [train_split, val_split, 0]
            test_split = [0, 0, 500]
            exp_rod = get_exp_rod(data_id = data_id)
            trainval_data_split, mesh, red_dims = loadexpwrapper(
                meshpath = exp_rod['meshpath'],
                datapath = exp_rod['trainval_datapath'],
                hidden_dims = exp_rod['hidden_dims'],
                split = trainval_split
            )
            test_data_split, mesh, red_dims = loadexpwrapper(
                meshpath = exp_rod['meshpath'],
                datapath = exp_rod['test_datapath'],
                hidden_dims = exp_rod['hidden_dims'],
                split = test_split
            )
            (utrain, uval, _) = trainval_data_split
            (_, _, utest) = test_data_split
        
            pod_error_analysis['test'][data_id-1,split_idx] = \
                get_pod_error(utrain = utrain, utest = utest, N = red_dims[-1])
            pod_error_analysis['train'][data_id-1,split_idx] = \
                get_pod_error(utrain = utrain, utest = utrain, N = red_dims[-1])
            pod_error_analysis['ns_trainval'][data_id-1,split_idx] = train_split + val_split

    return pod_error_analysis


### Generalization error (+ showcase how to load weights)

In [ ]:
def get_gen_error_raw_analysis(
    raw_test_error_analysis : dict
):

    # Initialize activation
    sharpness = 3
    from scipy.optimize import bisect
    alphaLeakyRelu = lambda alpha: LeakyReLU(alpha, 1.25)
    alpha_leaky = bisect(
        lambda alpha: (alphaLeakyRelu(alpha).sharpness - sharpness), 
        a = 1.3,
        b = 15
    )
    bilipactivation = LeakyReLU(alpha = alpha_leaky, beta = 1.25)
    bilipactivation.setup()

    # Initialize analysis dictionary
    raw_gen_error_analysis = dict()
    raw_gen_error_analysis['mse_train'] = list()
    raw_gen_error_analysis['mse_test'] = list()
    raw_gen_error_analysis['model_name'] = list()
    raw_gen_error_analysis['initialization'] = list()
    raw_gen_error_analysis['ns_train'] = list()
    raw_gen_error_analysis['ns_val'] = list()
    raw_gen_error_analysis['data_id'] = list()
    raw_gen_error_analysis['ns_trainval'] = list()

    # Setup device
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

    # Loop over architectures
    config = zip(
          list(raw_test_error_analysis['model_name']),
          list(raw_test_error_analysis['initialization']),
          list(raw_test_error_analysis['ns_train']),
          list(raw_test_error_analysis['ns_val']),
          list(raw_test_error_analysis['data_id'])
    )

    for model_name, initialization, ns_train, ns_val, dataid in config:
        
        # Set checkpoint path
        model_full_string = 'rod_' + model_name + '_' + \
            initialization + '_leaky_sharp=' + str(sharpness) + \
            '_train' + str(ns_train) + '_val' + str(ns_val) + \
            '_dataid' + str(dataid) + '.pt'
        ckpt_path = os.path.join(
            resultspath, 'ckpts', 'sample_size_robustness_ckpts', 
            model_full_string
        )

        # Get data split
        exp_rod = get_exp_rod(data_id = dataid)
        trainval_split = [ns_train, ns_val, 0]
        test_split = [0, 0, 500]
        trainval_data_split, mesh, red_dims = loadexpwrapper(
            meshpath = exp_rod['meshpath'],
            datapath = exp_rod['trainval_datapath'],
            hidden_dims = exp_rod['hidden_dims'],
            split = trainval_split
        )
        test_data_split, mesh, red_dims = loadexpwrapper(
            meshpath = exp_rod['meshpath'],
            datapath = exp_rod['test_datapath'],
            hidden_dims = exp_rod['hidden_dims'],
            split = test_split
        )
        (utrain, uval, _) = trainval_data_split
        (_, _, utest) = test_data_split
        normalizer = Normalizer(utrain)

        # Instantiate architectures
        if model_name == 'ae':
            model = AE(red_dims, bilipactivation)
        if model_name == 'sae':
            model = SAE(red_dims, bilipactivation)
        if model_name == 'sbae':
            model = SBAE(red_dims, bilipactivation)
        if model_name == 'soae':
            model = SOAE(red_dims, bilipactivation)

        # Initialize architectures
        if initialization == 'standard':
            model.standard()
        elif initialization == 'eys':
            nested_pod = NestedPOD(
                snapshots = normalizer.forward(utrain), 
                red_dims = red_dims, 
                bilipactivation = bilipactivation
            )
            model.eys(nested_pod = nested_pod)

        # Load weights
        model.load(ckpt_path)
        model.to(device = device)

        # Compute training and test errors
        train_pred = normalizer.backward(
            model(normalizer.forward(utrain))
        )
        test_pred = normalizer.backward(
            model(normalizer.forward(utest))
        )
        test_mse = metrics.mse(utrue = utest, upred = test_pred).item()
        train_mse = metrics.mse(utrue = utrain, upred = train_pred).item()

        # Store information in analysis dictionary
        raw_gen_error_analysis['mse_train'].append(train_mse)
        raw_gen_error_analysis['mse_test'].append(test_mse)
        raw_gen_error_analysis['model_name'].append(model_name)
        raw_gen_error_analysis['initialization'].append(initialization)
        raw_gen_error_analysis['ns_train'].append(ns_train)
        raw_gen_error_analysis['ns_val'].append(ns_val)
        raw_gen_error_analysis['data_id'].append(dataid)
        raw_gen_error_analysis['ns_trainval'].append(ns_train + ns_val)

    return raw_gen_error_analysis

In [ ]:
def extract_info_gen_error_analysis(
    raw_gen_error_analysis : dict
):

    # Extract unique values
    ns_trainval_unique = np.unique(raw_gen_error_analysis['ns_trainval'])
    initialization_unique = np.unique(raw_gen_error_analysis['initialization'])
    model_name_unique = np.unique(raw_gen_error_analysis['model_name'])

    # Initialize dictionary to store analysis result
    gen_error_analysis = dict()
    for model_name in model_name_unique:
        gen_error_analysis[model_name] = dict()
        for initialization in initialization_unique:
            gen_error_analysis[model_name][initialization] = dict()
            gen_error_analysis[model_name][initialization]['gen_error'] = list()
            gen_error_analysis[model_name][initialization]['ns_trainval'] = ns_trainval_unique

    # Loop
    for model_name in model_name_unique:
        cond_model_name = (
            np.array(raw_gen_error_analysis['model_name']) == model_name)
        for initialization in initialization_unique:
            cond_initialization = (
                np.array(raw_gen_error_analysis['initialization']) == initialization)
            for ns_trainval in ns_trainval_unique:
                cond_ns_trainval = (
                    np.array(raw_gen_error_analysis['ns_trainval']) == ns_trainval
                )
                cond = cond_ns_trainval * cond_initialization * cond_model_name
                curr_mse_test = np.array(raw_gen_error_analysis['mse_test'])[cond]
                curr_mse_train = np.array(raw_gen_error_analysis['mse_train'])[cond]
                gen_error_analysis[model_name][initialization]['gen_error'].append(
                    np.median(np.abs(curr_mse_train - curr_mse_test)))
    
    return gen_error_analysis

In [ ]:
def generate_figure(
    test_error_analysis : dict,
    gen_error_analysis : dict,
    pod_error_analysis : dict
):
    
    # Initialize figure specs -------------------------------------------------#
    fig, axs = plt.subplots (1, 2, figsize = (13, 4))
    cmap = plt.get_cmap('tab10')

    # Plot test error analysis-------------------------------------------------#
    for color_idx, model_name in enumerate(test_error_analysis.keys()):
        for initialization in test_error_analysis[model_name].keys():
            ns_trainval = test_error_analysis[model_name][initialization]['ns_trainval']
            MSE_med = test_error_analysis[model_name][initialization]['MSE_med']
            MSE_min = test_error_analysis[model_name][initialization]['MSE_min']
            MSE_max = test_error_analysis[model_name][initialization]['MSE_max']
            ns_trainval = np.array(ns_trainval)
            MSE_med = np.array(MSE_med)
            MSE_min = np.array(MSE_min)
            MSE_max = np.array(MSE_max)
            axs[0].loglog(
                ns_trainval,
                MSE_med,
                label = model_name.upper() if initialization == 'eys' else None,
                linestyle = '-' if initialization == 'eys' else '--',
                color = cmap(color_idx),
                marker = 'o'
            )
            axs[0].fill_between(
                ns_trainval,
                MSE_min,
                MSE_max,
                color = cmap(color_idx),
                alpha = 0.1,
                edgecolor = None
            )
            
            axs[0].set_xlabel('$N_s^{train} + N_s^{val}$', fontsize = 14)
            axs[0].set_ylabel('$MSE_{test}$', fontsize = 14)

    # POD test error
    axs[0].loglog(
        [50,150,450,1350], 
        np.median(pod_error_analysis['test'], axis = 0), 
        color = 'k', linestyle = '-', 
        linewidth = 2, marker = 'o', 
        label = 'POD'
    )

    # Set lims
    axs[0].set_ylim([3e1, 2e4])

    # Plot gen error analysis -------------------------------------------------#
    for color_idx, model_name in enumerate(gen_error_analysis.keys()):
        for initialization in gen_error_analysis[model_name].keys():
            ns_trainval = gen_error_analysis[model_name][initialization]['ns_trainval']
            relative_generalization_gap = np.array(
                gen_error_analysis[model_name][initialization]['gen_error']
            )
            axs[1].loglog(
                ns_trainval,
                relative_generalization_gap,
                linestyle = '-' if initialization == 'eys' else '--',
                color = cmap(color_idx),
                marker = 'o'
            )
            axs[1].set_xlabel('$N_s^{train} + N_s^{val}$', fontsize = 14)
            axs[1].set_ylabel('$|MSE_{test}- MSE_{train}|$', fontsize = 14)


    axs[1].loglog(
        pod_error_analysis['ns_trainval'][0,:], 
        np.median(np.abs(pod_error_analysis['test'] - \
                        pod_error_analysis['train']), axis = 0), 
        color = 'k', linestyle = '-', 
        linewidth = 2, marker = 'o'
    )

    axs[1].xaxis.set_tick_params(labelsize = 13)
    axs[1].yaxis.set_tick_params(labelsize = 13)
    axs[1].tick_params(axis='x', which = 'both', colors='grey')
    axs[1].tick_params(axis='y', which = 'both', colors='grey')
    axs[1].grid(
        visible = 'True', 
        which = 'both', 
        color = 'gray', 
        alpha = 0.3, 
        linestyle = ':'
    )

    # Modify figure specs -----------------------------------------------------#

    # Set lims & ticks
    for ax in axs:
        ax.xaxis.set_tick_params(labelsize = 13)
        ax.yaxis.set_tick_params(labelsize = 13)
        ax.tick_params(axis = 'both', which = 'both', colors = 'gray')
        ax.grid(
            visible = 'True', 
            which = 'both', 
            color = 'gray', 
            alpha = 0.3, 
            linestyle = ':'
        )

    # Set common legend to both plots
    plt.figlegend(
        bbox_to_anchor = (0.2, 0.95, 0.6, 0.12), 
        loc = 'lower left', 
        mode = 'expand', 
        ncols = 5, 
        borderaxespad = 0,
        fontsize = 12
    )

    # Save figure
    plt.subplots_adjust(wspace = 0.5)
    savepath = os.path.join('..', 'results', 'figures')
    if not os.path.exists(savepath):
        os.makedirs(savepath)
    plt.savefig(
        os.path.join(savepath, 'sample_size_robustness_full.png'),
        bbox_inches='tight'
    )

In [ ]:
# Full pipeline
raw_test_error_analysis = get_test_error_raw_analysis()
test_error_analysis = extract_info_test_error_analysis(
    raw_test_error_analysis = raw_test_error_analysis
)
pod_error_analysis = get_pod_error_analysis(
    raw_test_error_analysis = raw_test_error_analysis
)
raw_gen_error_analysis = get_gen_error_raw_analysis(
    raw_test_error_analysis = raw_test_error_analysis
)
gen_error_analysis = extract_info_gen_error_analysis(
    raw_gen_error_analysis = raw_gen_error_analysis
)

# Check consistency
assert np.sum(np.array(raw_gen_error_analysis['mse_test']) - np.array(raw_test_error_analysis['mse'])) == 0

In [ ]:
# Generate figure
generate_figure(
    test_error_analysis = test_error_analysis,
    gen_error_analysis = gen_error_analysis,
    pod_error_analysis = pod_error_analysis,
)